# 09_02 Choosing a bias-neutral reallocation under a per-class calibration constraint

`09_01` traced the row-grain WAPE↔bias frontier with two decision rules on the
persisted two-stage forecasts — global scaling and zeroing rows below an
occurrence-probability threshold — and both buy WAPE only by going biased low
(the near-L1 zeroing endpoint reaches 52.8% at **−9.4% bias**). The near-zero-bias
requirement therefore appeared to wall those points off.

This notebook tests what `09_01` did not: **take forecast mass away from
low-probability rows and put it back onto the surviving rows within a scope**,
so scoped forecast totals — and therefore bias at every grain at or above that
scope — are preserved *by construction*.

The rule used here is a **smooth reallocation**: each row keeps a weight that
fades in quadratically with its occurrence probability, so near-certain zeros
give up almost all their mass while borderline rows give up little. (A hard
variant — zero every row below a threshold and rescale the rest — is kept below
as a reference point; the smooth weight is its generalisation.)

The L1 argument for why any such rule should win: shrinking a row with
occurrence probability $p<\tfrac12$ toward zero reduces expected absolute
error, while adding a small increment to a dense row sitting near its
conditional median costs only second order. If that asymmetry is real, part of
the frontier's ~4 pp becomes reachable at **unchanged** pooled bias.

## Why the scale $t$ is not chosen by WAPE

Pooled bias is invariant to $t$ under the global scope — the renormalisation
pins it exactly. A tuning criterion that watches only row WAPE and pooled bias
is therefore watching a constraint that *cannot bind*, and it will run $t$ up
to wherever WAPE bottoms out. That is what an earlier revision of this notebook
did, selecting $t^\*=0.60$.

What such a rule actually does is move forecast mass **between** intermittency
classes. So the binding question is **conditional** bias: is
$E[\hat y - y \mid \text{class}] = 0$ for classes knowable at forecast time?
This revision therefore:

1. establishes whether the baseline model is conditionally biased on the frozen
   ADI classes at all — testing the aggregate class biases against their
   origin-to-origin sampling error, not against zero by eye;
2. selects $t$ under a **calibration constraint** — the largest $t$ whose
   induced change in class bias stays inside one sampling standard error — with
   the constraint evaluated on the tune half and the result frozen and scored on
   the test half;
3. reports what the WAPE-optimal $t$ would have cost in calibration.

Conditioning classes are the frozen ADI terciles of `09_01`, computed from demand
history strictly before **2026-02-02** — 28 days before the first origin — so no
evaluation-period information enters the partition. Segmentations built from
demand *realized during* the evaluation window (e.g. banding on the occurrence
rate actually observed over the 20 origins) condition on the outcome and are not
used: a series lands in such a "sparse" band partly because its actuals came in
below its own norm, which manufactures apparent overforecasting.

No model is trained or refit; every number uses the persisted two-stage
forecasts of the 20 evaluation origins — the adopted `08_04` mainline on
`transactions_fixed` (row 56.92%, pooled bias −0.25%).

In [1]:
from pathlib import Path
import sys

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.benchmark import load_benchmark_design
from src.models.lightgbm import TWO_STAGE_MODEL_NAME
from src.models.results import result_path

pd.set_option('display.max_columns', 40)

design = load_benchmark_design()
forecast_path = result_path(TWO_STAGE_MODEL_NAME, design)
con = duckdb.connect()
con.execute('PRAGMA threads=4')
con.execute(
    '''
    CREATE OR REPLACE TEMP TABLE rows AS
    SELECT ARTIKEL_ID::BIGINT AS ARTIKEL_ID, MARKT_ID::BIGINT AS MARKT_ID,
           CAST(origin AS DATE) AS origin, CAST(period AS DATE) AS period,
           actual::DOUBLE AS actual, forecast::DOUBLE AS forecast,
           occurrence_probability::DOUBLE AS p_occ
    FROM read_csv_auto(?)
    WHERE is_active
    ''',
    [str(forecast_path)],
)
overview = con.execute('''
    SELECT COUNT(*) AS active_rows, COUNT(DISTINCT origin) AS origins,
           AVG((actual > 0)::INTEGER) AS occurrence_rate,
           AVG((p_occ < 0.5)::INTEGER) AS rows_with_p_below_half
    FROM rows
''').fetchdf()
display(overview.style.format({
    'active_rows': '{:,.0f}', 'origins': '{:,.0f}',
    'occurrence_rate': '{:.1%}', 'rows_with_p_below_half': '{:.1%}',
}))

,active_rows,origins,occurrence_rate,rows_with_p_below_half
0,"2,498,967",20,38.0%,67.4%


## Frozen ADI classes

The conditioning partition, established once, before anything else uses it.
ADI = operational days per positive-demand day, computed per series from history
strictly before the cutoff. A1 = lowest ADI (densest) … A3 = highest (sparsest);
series with fewer than 5 positive pre-cutoff days are `unclassified`.

In [2]:
CLASSIFICATION_CUTOFF = pd.Timestamp('2026-02-02')
assert CLASSIFICATION_CUTOFF < design.first_origin, 'classes must be frozen before the first origin'

series_keys = con.execute('SELECT DISTINCT ARTIKEL_ID, MARKT_ID FROM rows').fetchdf()
con.register('series_keys', series_keys)
frozen_history = con.execute('''
    WITH daily AS (
        SELECT d.ARTIKEL_ID::BIGINT AS ARTIKEL_ID, d.MARKT_ID::BIGINT AS MARKT_ID,
               CAST(d.DATE AS DATE) AS period,
               SUM(CAST(COALESCE(d.ABVERKAUFTE_MENGE_KG, 0) AS DOUBLE)) AS demand
        FROM read_parquet(?) AS d
        JOIN series_keys AS s
          ON d.ARTIKEL_ID = s.ARTIKEL_ID AND d.MARKT_ID = s.MARKT_ID
        WHERE d.is_active AND (d.is_fcm OR d.is_pseudo)
          AND d.WGR_ID IN (890, 900) AND CAST(d.DATE AS DATE) < ?
        GROUP BY 1, 2, 3
    )
    SELECT ARTIKEL_ID, MARKT_ID, COUNT_IF(demand > 0) AS n_pos,
           COUNT(*)::DOUBLE / NULLIF(COUNT_IF(demand > 0), 0) AS ADI
    FROM daily GROUP BY 1, 2
''', [str(design.data_dir / '*.parquet'), CLASSIFICATION_CUTOFF.date()]).fetchdf()

regimes = series_keys.merge(frozen_history, on=['ARTIKEL_ID', 'MARKT_ID'], how='left')
eligible = regimes.n_pos.fillna(0).ge(5)
assert eligible.sum() == 21_140          # same frozen classes as 09_01 (transactions_fixed)
regimes['adi_bin'] = 'unclassified'
regimes.loc[eligible, 'adi_bin'] = pd.qcut(
    regimes.loc[eligible, 'ADI'], 3, labels=['A1', 'A2', 'A3']).astype(str)
con.register('adi', regimes[['ARTIKEL_ID', 'MARKT_ID', 'adi_bin']])

CLASSES = ['A1', 'A2', 'A3']
composition = con.execute('''
    SELECT adi_bin,
           COUNT(DISTINCT (ARTIKEL_ID, MARKT_ID)) AS series,
           SUM(actual) / (SELECT SUM(actual) FROM rows) AS volume_share,
           AVG((actual > 0)::INTEGER) AS occurrence_rate,
           AVG(p_occ) AS mean_p_occ
    FROM rows JOIN adi USING (ARTIKEL_ID, MARKT_ID)
    GROUP BY 1 ORDER BY 1
''').fetchdf()
display(composition.style.format({'series': '{:,.0f}', 'volume_share': '{:.2%}',
                                  'occurrence_rate': '{:.1%}', 'mean_p_occ': '{:.3f}'}))

,adi_bin,series,volume_share,occurrence_rate,mean_p_occ
0,A1,"7,051",72.98%,66.1%,0.666
1,A2,"7,043",17.77%,29.7%,0.301
2,A3,"7,046",8.85%,18.4%,0.183
3,unclassified,782,0.40%,33.7%,0.340


## Method

The rule has two parameters — a scale $t$ and a redistribution **scope** $S$:

1. every row gets the weight $h_i = \min\!\left(1, (p_i/t)^2\right)$, where
   $p_i$ is the model's persisted occurrence probability — rows with
   $p_i \ge t$ are untouched, rows far below $t$ fade toward zero;
2. weighted forecasts are renormalised per scope,
   $f_i' = f_i\,h_i \cdot \frac{\sum_S f}{\sum_S f h}$, so each scope's
   forecast sum is exactly unchanged;
3. **guard**: a scope whose weighted sum is zero is left untouched, which keeps
   totals exact.

The hard threshold rule (zero below $t$, rescale the rest) is the limiting case
of this weight as the exponent → ∞; with the quadratic exponent the rule takes
a lot from near-certain zeros and only a little from borderline rows.

The scope decides which aggregates *cannot* be harmed:

| scope | mass moves across | preserved by construction |
|---|---|---|
| origin (global) | articles, stores, days | pooled total per origin → pooled bias |
| article-week | stores, days within an article-week | article-week grain and above |
| article-store-week | days within a series-week | article-store-week grain and above |

The rule is a pure post-hoc decision layer: it uses nothing that is not already
available at forecast time.

In [3]:
SCOPES = {
    'origin (global)': 'origin',
    'article-week': 'ARTIKEL_ID, origin',
    'article-store-week': 'ARTIKEL_ID, MARKT_ID, origin',
}
GRAINS = {
    'row': None,
    'article-store-week': 'ARTIKEL_ID, MARKT_ID, origin',
    'article-day': 'ARTIKEL_ID, origin, period',
    'article-week': 'ARTIKEL_ID, origin',
}


def adjusted_sql(t, scope_keys, origin_filter=''):
    """Smooth reallocation: weight h = min(1, (p/t)^2), renormalised per scope."""
    return f'''
        WITH weighted AS (
            SELECT *, forecast * LEAST(1.0, POW(p_occ / {t}, 2)) AS fh
            FROM rows {origin_filter}
        ),
        scoped AS (
            SELECT *,
                   SUM(forecast) OVER (PARTITION BY {scope_keys}) AS tot,
                   SUM(fh) OVER (PARTITION BY {scope_keys}) AS hsum
            FROM weighted
        )
        SELECT ARTIKEL_ID, MARKT_ID, origin, period, actual,
               CASE WHEN hsum IS NULL OR hsum <= 0 THEN forecast
                    ELSE fh * tot / hsum END AS forecast
        FROM scoped
    '''


def hard_sql(t, scope_keys, origin_filter=''):
    """Reference: hard threshold (zero below t, rescale the rest)."""
    return f'''
        WITH scoped AS (
            SELECT *,
                   SUM(forecast) OVER (PARTITION BY {scope_keys}) AS tot,
                   SUM(CASE WHEN p_occ >= {t} THEN forecast END)
                       OVER (PARTITION BY {scope_keys}) AS kept
            FROM rows {origin_filter}
        )
        SELECT ARTIKEL_ID, MARKT_ID, origin, period, actual,
               CASE WHEN kept IS NULL OR kept <= 0 THEN forecast
                    WHEN p_occ < {t} THEN 0
                    ELSE forecast * tot / kept END AS forecast
        FROM scoped
    '''


def score(table_sql):
    out = {}
    for grain, keys in GRAINS.items():
        inner = (f'SELECT actual, forecast FROM ({table_sql})' if keys is None else
                 f'SELECT SUM(actual) AS actual, SUM(forecast) AS forecast '
                 f'FROM ({table_sql}) GROUP BY {keys}')
        wape, bias = con.execute(
            f'SELECT SUM(ABS(forecast - actual)) / SUM(actual), '
            f'SUM(forecast - actual) / SUM(actual) FROM ({inner})').fetchone()
        out[grain] = wape
        if grain == 'row':
            out['bias'] = bias
    return out


baseline = score('SELECT * FROM rows')
display(pd.DataFrame([{'forecast': 'two-stage model (baseline)', **baseline}])
        .style.format({g: '{:.2%}' for g in GRAINS} | {'bias': '{:+.2%}'}))

,forecast,row,bias,article-store-week,article-day,article-week
0,two-stage model (baseline),56.92%,-0.25%,36.40%,23.41%,19.88%


In [4]:
def class_bias(table_sql):
    'Relative bias per frozen ADI class, pooled over the rows supplied.'
    return dict(con.execute(f"""
        SELECT adi_bin, SUM(forecast - actual) / SUM(actual)
        FROM ({table_sql}) JOIN adi USING (ARTIKEL_ID, MARKT_ID)
        GROUP BY 1""").fetchall())


def class_bias_per_origin(table_sql):
    'Per-origin class bias -> the sampling spread the aggregate is measured against.'
    return con.execute(f"""
        SELECT origin, adi_bin, SUM(forecast - actual) / SUM(actual) AS bias
        FROM ({table_sql}) JOIN adi USING (ARTIKEL_ID, MARKT_ID)
        GROUP BY 1, 2""").fetchdf().pivot(index='origin', columns='adi_bin', values='bias')

## Sweep: scale × scope

WAPE at four grains for every (scope, $t$) combination, with the row-grain delta
against the baseline in percentage points. Bias is reported once — the rule
preserves it identically at every scope.

In [5]:
sweep = []
for scope_name, keys in SCOPES.items():
    for t in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
        sweep.append({'scope': scope_name, 't': t, **score(adjusted_sql(t, keys))})
sweep = pd.DataFrame(sweep)
for g in GRAINS:
    sweep[f'Δ {g} (pp)'] = (sweep[g] - baseline[g]) * 100
display(sweep[['scope', 't', 'bias', 'row', 'Δ row (pp)',
               'article-store-week', 'Δ article-store-week (pp)',
               'article-day', 'Δ article-day (pp)',
               'article-week', 'Δ article-week (pp)']]
        .style.format({'t': '{:.2f}', 'bias': '{:+.2%}',
                       **{g: '{:.2%}' for g in GRAINS},
                       **{f'Δ {g} (pp)': '{:+.2f}' for g in GRAINS}})
        .background_gradient(subset=['Δ row (pp)'], cmap='RdYlGn_r'))

,scope,t,bias,row,Δ row (pp),article-store-week,Δ article-store-week (pp),article-day,Δ article-day (pp),article-week,Δ article-week (pp)
0,origin (global),0.10,-0.25%,56.46%,-0.45,36.07%,-0.33,23.37%,-0.04,19.86%,-0.02
1,origin (global),0.20,-0.25%,55.88%,-1.03,35.86%,-0.54,23.55%,+0.14,20.10%,+0.22
2,origin (global),0.30,-0.25%,55.38%,-1.53,35.84%,-0.55,23.94%,+0.53,20.59%,+0.71
3,origin (global),0.40,-0.25%,55.01%,-1.90,35.99%,-0.40,24.51%,+1.09,21.26%,+1.38
4,origin (global),0.50,-0.25%,54.78%,-2.14,36.30%,-0.10,25.25%,+1.84,22.08%,+2.21
5,origin (global),0.60,-0.25%,54.69%,-2.23,36.76%,+0.36,26.21%,+2.79,23.12%,+3.24
6,origin (global),0.70,-0.25%,54.74%,-2.18,37.40%,+1.00,27.38%,+3.97,24.37%,+4.49
7,origin (global),0.80,-0.25%,54.95%,-1.97,38.23%,+1.84,28.82%,+5.41,25.94%,+6.06
8,article-week,0.10,-0.25%,56.69%,-0.22,36.25%,-0.15,23.43%,+0.02,19.88%,-0.00
9,article-week,0.20,-0.25%,56.38%,-0.53,36.20%,-0.20,23.51%,+0.10,19.88%,+0.00


## Reading the sweep

- **The global (per-origin) scope dominates.** Row WAPE falls monotonically to
  $t≈0.60$ with pooled bias pinned at the baseline value. That monotonicity is
  the warning, not the result: pooled bias is invariant to $t$ under this scope,
  so nothing in this table can stop $t$ from running away. What the row column is
  measuring at the aggressive end is the metric's median-seeking geometry, not a
  better forecast.
- **A conservative band improves the row and weekly grains at once**: at
  $t=0.10$ every grain improves simultaneously; by $t=0.20$ the row and
  article-store-week grains keep gaining while article-day begins to pay.
- **The gain requires moving mass *across* series.** The article-store-week
  scope — pure within-week reallocation — is nearly inert. This confirms the
  `09_01` allocation-ladder finding from the opposite direction: within-week
  allocation is already near-optimal; what the model leaves on the table is
  *cross-series* mass placement. It also means the lever is precisely the one
  that redistributes between intermittency classes, which is why the next two
  sections check what that does to per-class calibration before choosing $t$.
- The article-week scope sits in between and leaves the article-week grain
  untouched by construction — the conservative variant if article-level totals
  are a hard reporting constraint.

## Is the baseline conditionally biased at all?

Before tuning a rule that redistributes mass between classes, the premise has to
be checked: does the model *have* a per-class bias to correct? The aggregate
class biases below are pooled over all 20 origins. They are compared against the
origin-to-origin spread of the same quantity — a class bias of +4% means nothing
if the weekly values swing ±20%.

In [6]:
per_origin_bias = class_bias_per_origin('SELECT * FROM rows')
agg = class_bias('SELECT * FROM rows')

calibration = pd.DataFrame([
    {'class': c,
     'aggregate bias': agg[c],
     'per-origin sd': per_origin_bias[c].std(),
     'per-origin min': per_origin_bias[c].min(),
     'per-origin max': per_origin_bias[c].max(),
     'se of aggregate': per_origin_bias[c].std() / np.sqrt(len(per_origin_bias)),
     't-stat': agg[c] / (per_origin_bias[c].std() / np.sqrt(len(per_origin_bias)))}
    for c in CLASSES])
display(calibration.style.format({
    'aggregate bias': '{:+.2%}', 'per-origin sd': '{:.2%}',
    'per-origin min': '{:+.2%}', 'per-origin max': '{:+.2%}',
    'se of aggregate': '{:.2%}', 't-stat': '{:+.2f}'}).hide(axis='index'))

display(per_origin_bias[CLASSES].style.format('{:+.2%}')
        .set_caption('baseline relative bias per origin and frozen ADI class'))

class,aggregate bias,per-origin sd,per-origin min,per-origin max,se of aggregate,t-stat
A1,-1.20%,12.14%,-21.70%,+31.38%,2.71%,-0.44
A2,+1.29%,17.38%,-29.26%,+41.16%,3.89%,+0.33
A3,+3.84%,16.43%,-22.26%,+42.14%,3.67%,+1.04


adi_bin,A1,A2,A3
origin,,,
2026-03-02 00:00:00,-5.49%,+12.97%,+12.58%
2026-03-09 00:00:00,+12.89%,+21.32%,+15.33%
2026-03-16 00:00:00,+0.84%,+2.56%,+8.86%
2026-03-23 00:00:00,+8.29%,+10.20%,+16.95%
2026-03-30 00:00:00,-10.66%,-21.86%,-10.54%
2026-04-06 00:00:00,+8.25%,+30.36%,+15.43%
2026-04-13 00:00:00,+31.38%,+41.16%,+42.14%
2026-04-20 00:00:00,-6.08%,-9.55%,-16.54%
2026-04-27 00:00:00,-21.70%,-29.26%,-18.13%


**The baseline is conditionally unbiased on these classes, within sampling error.**
No class aggregate reaches one standard error of zero (|t| ≤ 1.04). The per-origin
values swing from roughly −22% to +42% and move together across classes — these are
event and post-event weeks, the systematic weekly-total error `09_01` Addendum 3
attributed to the calendar programme, not a class-specific tilt.

Two consequences for what follows:

1. **The rule cannot be justified as a bias correction.** There is no measurable
   per-class bias for it to remove.
2. **The relevant risk is the opposite one** — that the rule *introduces* a class
   bias. That change is measured on the same origins as a paired difference, so it
   carries far less noise than the levels do and can be resolved precisely.

## Choosing $t$ under a calibration constraint

The honest protocol: everything that selects $t$ uses origins 1–10 only; the
chosen value is frozen and scored on origins 11–20.

**Criterion.** Take the largest $t$ whose *induced change* in class bias stays
within one standard error of the class-bias estimate on the tune half — i.e. the
rule may not perturb calibration by more than the noise with which calibration
itself is measured. Because the induced change is a paired per-origin difference
it is estimated far more tightly than the levels, so this is a real constraint
and not a formality.

The WAPE-optimal $t$ is reported alongside, to price what the constraint costs.

In [7]:
origins = [r[0] for r in con.execute(
    'SELECT DISTINCT origin FROM rows ORDER BY origin').fetchall()]
split = origins[9]
tune_f, test_f = f"WHERE origin <= DATE '{split}'", f"WHERE origin > DATE '{split}'"

tune_per_origin = class_bias_per_origin(f'SELECT * FROM rows {tune_f}')
tune_se = {c: tune_per_origin[c].std() / np.sqrt(len(tune_per_origin)) for c in CLASSES}
print('tune-half class-bias standard error: '
      + ', '.join(f'{c} {tune_se[c]:.2%}' for c in CLASSES))

grid = np.round(np.arange(0.05, 0.86, 0.05), 2)
tune_rows = []
for t in grid:
    adjusted = adjusted_sql(t, 'origin', tune_f)
    delta = con.execute(f'''
        WITH adj AS ({adjusted})
        SELECT ad.adi_bin,
               SUM(adj.forecast - adj.actual) / SUM(adj.actual)
             - SUM(rows.forecast - rows.actual) / SUM(rows.actual) AS d
        FROM adj JOIN rows USING (ARTIKEL_ID, MARKT_ID, origin, period)
        JOIN adi AS ad USING (ARTIKEL_ID, MARKT_ID)
        GROUP BY 1''').fetchdf().set_index('adi_bin')['d'].to_dict()
    tune_rows.append({
        't': t,
        'tune row WAPE': con.execute(
            f'SELECT SUM(ABS(forecast - actual)) / SUM(actual) FROM ({adjusted})').fetchone()[0],
        **{f'Δ bias {c}': delta[c] for c in CLASSES},
        'within 1 se': all(abs(delta[c]) <= tune_se[c] for c in CLASSES)})
tune = pd.DataFrame(tune_rows)

t_star = float(tune.loc[tune['within 1 se'], 't'].max())
t_wape = float(tune.loc[tune['tune row WAPE'].idxmin(), 't'])
display(tune.style.format({'t': '{:.2f}', 'tune row WAPE': '{:.2%}'}
                          | {f'Δ bias {c}': '{:+.2%}' for c in CLASSES})
        .hide(axis='index'))
print(f'calibration-constrained  t* = {t_star:.2f}')
print(f'WAPE-optimal (unconstrained) t = {t_wape:.2f}')

test_rows = []
for label, sql in [
    ('baseline', f'SELECT * FROM rows {test_f}'),
    (f'smooth t* = {t_star:.2f} (calibration-constrained)', adjusted_sql(t_star, 'origin', test_f)),
    (f'smooth t = {t_wape:.2f} (WAPE-optimal)', adjusted_sql(t_wape, 'origin', test_f)),
    ('reference: hard threshold t = 0.40', hard_sql(0.40, 'origin', test_f)),
]:
    cb = class_bias(sql)
    test_rows.append({'forecast (origins 11-20)': label, **score(sql),
                      **{f'bias {c}': cb[c] for c in CLASSES}})
display(pd.DataFrame(test_rows)
        .style.format({g: '{:.2%}' for g in GRAINS} | {'bias': '{:+.2%}'}
                      | {f'bias {c}': '{:+.2%}' for c in CLASSES}))

tune-half class-bias standard error: A1 4.73%, A2 7.09%, A3 5.97%


t,tune row WAPE,Δ bias A1,Δ bias A2,Δ bias A3,within 1 se
0.05,57.00%,+0.19%,-0.16%,-1.23%,True
0.10,56.72%,+0.58%,-0.57%,-3.46%,True
0.15,56.40%,+1.06%,-1.26%,-5.98%,False
0.20,56.09%,+1.61%,-2.20%,-8.44%,False
0.25,55.80%,+2.17%,-3.36%,-10.68%,False
0.30,55.55%,+2.74%,-4.65%,-12.63%,False
0.35,55.34%,+3.27%,-5.99%,-14.27%,False
0.40,55.17%,+3.77%,-7.34%,-15.59%,False
0.45,55.04%,+4.22%,-8.64%,-16.62%,False
0.50,54.94%,+4.62%,-9.85%,-17.41%,False


calibration-constrained  t* = 0.10
WAPE-optimal (unconstrained) t = 0.60


,forecast (origins 11-20),row,bias,article-store-week,article-day,article-week,bias A1,bias A2,bias A3
0,baseline,56.65%,-1.26%,35.53%,22.85%,18.60%,-2.03%,-0.16%,+1.66%
1,smooth t* = 0.10 (calibration-constrained),56.19%,-1.26%,35.17%,22.76%,18.52%,-1.47%,-0.89%,-1.54%
2,smooth t = 0.60 (WAPE-optimal),54.52%,-1.26%,35.68%,25.35%,21.32%,+2.31%,-8.77%,-16.15%
3,reference: hard threshold t = 0.40,54.38%,-1.26%,35.84%,25.38%,21.31%,+2.53%,-9.13%,-17.30%


In [8]:
per_origin = con.execute(f'''
    WITH adj AS ({adjusted_sql(t_star, 'origin')})
    SELECT adj.origin,
           SUM(ABS(rows.forecast - rows.actual)) / SUM(rows.actual) AS wape_base,
           SUM(ABS(adj.forecast - adj.actual)) / SUM(adj.actual) AS wape_rule
    FROM adj JOIN rows USING (ARTIKEL_ID, MARKT_ID, origin, period)
    GROUP BY 1 ORDER BY 1
''').fetchdf()
per_origin['delta_pp'] = (per_origin.wape_rule - per_origin.wape_base) * 100
print(f'origins improved at t* = {t_star:.2f}: '
      f'{(per_origin.delta_pp < 0).sum()} / {len(per_origin)}   '
      f'(delta range {per_origin.delta_pp.min():+.2f} to {per_origin.delta_pp.max():+.2f} pp)')
display(per_origin.style.format({'wape_base': '{:.2%}', 'wape_rule': '{:.2%}',
                                 'delta_pp': '{:+.2f}'}))

origins improved at t* = 0.10: 20 / 20   (delta range -0.60 to -0.25 pp)


,origin,wape_base,wape_rule,delta_pp
0,2026-03-02 00:00:00,54.92%,54.40%,-0.51
1,2026-03-09 00:00:00,59.60%,59.25%,-0.35
2,2026-03-16 00:00:00,55.93%,55.43%,-0.49
3,2026-03-23 00:00:00,59.93%,59.46%,-0.48
4,2026-03-30 00:00:00,48.49%,48.12%,-0.37
5,2026-04-06 00:00:00,58.57%,58.19%,-0.38
6,2026-04-13 00:00:00,67.90%,67.65%,-0.25
7,2026-04-20 00:00:00,58.89%,58.35%,-0.54
8,2026-04-27 00:00:00,57.05%,56.55%,-0.50
9,2026-05-04 00:00:00,55.03%,54.43%,-0.60


## What the rule actually does

Mass accounting at $t^\*$: how many rows are touched, how much forecast mass
actually moves, and how much realized volume sat on the rows that gave it up.


In [9]:
diag = con.execute(f'''
    WITH w AS (SELECT *, LEAST(1.0, POW(p_occ / {t_star}, 2)) AS h FROM rows)
    SELECT AVG((h < 1)::INTEGER) AS rows_downweighted,
           SUM(forecast * (1 - h)) / SUM(forecast) AS forecast_mass_taken,
           SUM(actual * (1 - h)) / SUM(actual) AS actual_volume_weight_taken,
           SUM(forecast) / SUM(forecast * h) AS renormalisation_factor
    FROM w
''').fetchdf()
display(diag.style.format('{:.2%}', subset=['rows_downweighted', 'forecast_mass_taken',
                                            'actual_volume_weight_taken'])
        .format({'renormalisation_factor': '{:.4f}'}))

,rows_downweighted,forecast_mass_taken,actual_volume_weight_taken,renormalisation_factor
0,0.305720,0.006119,0.005535,1.0062


## Effect per frozen ADI class

In [10]:
class_rows = []
for t in (t_star, t_wape):
    df = con.execute(f'''
        WITH adj AS ({adjusted_sql(t, 'origin')})
        SELECT '{t:.2f}' AS t, adi_bin,
               SUM(rows.actual) / (SELECT SUM(actual) FROM rows) AS volume_share,
               SUM(ABS(rows.forecast - rows.actual)) / SUM(rows.actual) AS wape_base,
               SUM(ABS(adj.forecast - adj.actual)) / SUM(adj.actual) AS wape_rule,
               SUM(rows.forecast - rows.actual) / SUM(rows.actual) AS bias_base,
               SUM(adj.forecast - adj.actual) / SUM(adj.actual) AS bias_rule
        FROM adj JOIN rows USING (ARTIKEL_ID, MARKT_ID, origin, period)
        JOIN adi USING (ARTIKEL_ID, MARKT_ID)
        GROUP BY 1, 2 ORDER BY 1, 2
    ''').fetchdf()
    class_rows.append(df)
class_table = pd.concat(class_rows, ignore_index=True)
class_table['Δ wape (pp)'] = (class_table.wape_rule - class_table.wape_base) * 100
class_table['bias / se'] = [
    row.bias_rule / (per_origin_bias[row.adi_bin].std() / np.sqrt(len(per_origin_bias)))
    if row.adi_bin in CLASSES else np.nan
    for row in class_table.itertuples()]
display(class_table.style.format({'volume_share': '{:.1%}', 'wape_base': '{:.2%}',
                                  'wape_rule': '{:.2%}', 'bias_base': '{:+.2%}',
                                  'bias_rule': '{:+.2%}', 'Δ wape (pp)': '{:+.2f}',
                                  'bias / se': '{:+.2f}'}))

,t,adi_bin,volume_share,wape_base,wape_rule,bias_base,bias_rule,Δ wape (pp),bias / se
0,0.10,A1,73.0%,48.31%,48.35%,-1.20%,-0.63%,+0.04,-0.23
1,0.10,A2,17.8%,76.54%,75.55%,+1.29%,+0.64%,-1.00,+0.16
2,0.10,A3,8.8%,86.48%,83.06%,+3.84%,+0.50%,-3.42,+0.14
3,0.10,unclassified,0.4%,100.75%,98.91%,+14.15%,+12.52%,-1.84,+nan
4,0.60,A1,73.0%,48.31%,48.70%,-1.20%,+3.61%,+0.39,+1.33
5,0.60,A2,17.8%,76.54%,69.19%,+1.29%,-9.03%,-7.35,-2.32
6,0.60,A3,8.8%,86.48%,73.40%,+3.84%,-14.25%,-13.08,-3.88
7,0.60,unclassified,0.4%,100.75%,87.49%,+14.15%,-6.10%,-13.26,+nan


**Reading the class table.** At $t^\*=0.10$ the rule leaves calibration where it
found it. Every class bias moves *toward* zero and lands well inside a quarter of
its own standard error — A1 −1.20% → −0.63%, A2 +1.29% → +0.64%, A3 +3.84% →
+0.50% (`bias / se` = −0.23, +0.16, +0.14). The accuracy gain is concentrated
where the probability weight lives: A3 −3.42 pp and A2 −1.00 pp of WAPE, with A1
— 73% of volume — flat at +0.04 pp. The mechanism is deliberately small: only
0.61% of forecast mass is displaced, renormalising by ×1.0062.

At the WAPE-optimal $t=0.60$ the same rule is a different object. It buys 2.2 pp
of row WAPE and pays in class bias: A2 +1.29% → **−9.03%** and A3 +3.84% →
**−14.25%**, i.e. −2.32 and −3.88 standard errors from zero, while A1 is pushed
to +3.61%. The baseline's class biases were not resolvable against origin-to-origin
noise; these are, comfortably. The 20 origins can tell that a $t=0.60$ forecast is
systematically short on the sparse classes, and they could not tell that of the
model itself.

That asymmetry is the whole argument. The rule at $t=0.60$ does not correct a
bias — there was none to correct — it manufactures one, and books the proceeds as
a WAPE improvement because pooled bias, the only bias the sweep table watches, is
pinned by construction.

The `unclassified` band (782 series, 0.4% of volume, fewer than 5 positive
pre-cutoff days) is the one segment where the baseline *is* badly calibrated:
+14.15%. It is too small to move any aggregate, and the rule only takes it to
+12.52% at $t^\*$, but it is the honest place to look for genuine overforecasting
of new and ultra-sparse series.

## Why the within-week distribution is not where the gain is

The article-store-week scope — reallocation strictly *within* a series-week —
was nearly inert in the sweep. A natural suspicion: are the weekdays simply
indistinguishable (equal occurrence probability), so there is nothing to move
between them? The tables below check the actual structure: the mean occurrence
probability and demand share per weekday and class, how the weights partition
series-weeks at $t^\*$, and how much forecast mass the rule can even *displace*
within series-weeks compared to the global scope.

In [11]:
wd = con.execute('''
    SELECT adi_bin, EXTRACT(ISODOW FROM period) AS weekday, AVG(p_occ) AS mean_p_occ,
           SUM(actual) / SUM(SUM(actual)) OVER (PARTITION BY adi_bin) AS demand_share
    FROM rows JOIN adi USING (ARTIKEL_ID, MARKT_ID)
    GROUP BY 1, 2 ORDER BY 1, 2
''').fetchdf()
display(wd.pivot(index='weekday', columns='adi_bin', values='mean_p_occ')
        .style.format('{:.3f}').set_caption('mean occurrence probability by weekday'))
display(wd.pivot(index='weekday', columns='adi_bin', values='demand_share')
        .style.format('{:.1%}').set_caption('share of class demand by weekday'))

composition = con.execute(f'''
    WITH wk AS (
        SELECT adi_bin, ARTIKEL_ID, MARKT_ID, origin,
               MIN(p_occ) AS pmin, MAX(p_occ) AS pmax
        FROM rows JOIN adi USING (ARTIKEL_ID, MARKT_ID)
        GROUP BY 1, 2, 3, 4
    )
    SELECT adi_bin,
           AVG((pmax < {t_star})::INTEGER) AS weeks_every_row_downweighted,
           AVG((pmin >= {t_star})::INTEGER) AS weeks_untouched
    FROM wk GROUP BY 1 ORDER BY 1
''').fetchdf()
display(composition.style.format('{:.1%}', subset=composition.columns[1:])
        .set_caption(f'series-week partition at t* = {t_star:.2f}'))

displaced = {}
for label, keys in [('global scope', 'origin'),
                    ('within series-week scope', 'ARTIKEL_ID, MARKT_ID, origin')]:
    displaced[label] = con.execute(f'''
        SELECT SUM(ABS(adj.forecast - rows.forecast)) / (2 * SUM(rows.forecast))
        FROM ({adjusted_sql(t_star, keys)}) AS adj
        JOIN rows USING (ARTIKEL_ID, MARKT_ID, origin, period)
    ''').fetchone()[0]
display(pd.DataFrame([displaced]).style.format('{:.2%}')
        .set_caption(f'forecast mass actually displaced at t* = {t_star:.2f}'))

adi_bin,A1,A2,A3,unclassified
weekday,,,,
1,0.612,0.249,0.146,0.275
2,0.613,0.252,0.148,0.278
3,0.627,0.264,0.159,0.305
4,0.661,0.292,0.176,0.324
5,0.728,0.355,0.221,0.411
6,0.755,0.392,0.246,0.443


adi_bin,A1,A2,A3,unclassified
weekday,,,,
1,11.9%,11.0%,10.4%,11.7%
2,12.5%,12.0%,11.5%,12.2%
3,14.0%,13.8%,13.5%,14.2%
4,16.0%,17.1%,17.0%,15.4%
5,19.0%,20.0%,20.9%,20.2%
6,26.5%,26.2%,26.7%,26.4%


,adi_bin,weeks_every_row_downweighted,weeks_untouched
0,A1,1.8%,90.9%
1,A2,18.8%,57.3%
2,A3,38.6%,29.9%
3,unclassified,13.1%,63.8%


,global scope,within series-week scope
0,0.61%,0.27%


**No — the weekdays are not equi-probable, and that is not the problem.**

1. **A strong weekday gradient exists and is already priced in.** Occurrence
   probability rises monotonically Monday → Saturday in every class (A1
   0.61 → 0.76, A3 0.15 → 0.25; Sunday is closed), and the demand share more
   than doubles (≈12% → ≈27%). The model knows this — `09_01`'s allocation
   ladder showed its within-week *shape* already beats the **true**
   (future-knowledge) static weekday profile. The weekly distribution is
   solved as far as it is solvable.
2. **Within a series-week the weights nearly cancel.** `p_occ` varies far more
   across series than across weekdays within a series. At $t^\*=0.10$, 38.6% of
   A3 series-weeks have *every* row down-weighted — a near-common factor that the
   renormalisation simply divides back out — while 90.9% of A1 series-weeks are
   untouched entirely. The mass the rule can actually displace within
   series-weeks is **0.27%** of the total forecast, versus **0.61%** under the
   global scope, and what little does move goes from a series' Monday to the
   same series' Saturday — a reshuffle along a shape that is already
   near-optimal (point 1), which is why it buys ≈0.
3. **What remains within the week is the realized draw, not a mislearned
   profile.** `09_01` measured it directly: even knowing the *true* weekly
   total and the *true* weekday profile scores 43.7% — the day-to-day
   dispersion around the profile (~50% CV under matched conditions) is the
   residual, and no reallocation rule can place it. The rule's gain therefore
   has to come — and does come — from moving mass **between** series, which only
   a scope spanning series can reach. That is also precisely why the choice of
   $t$ is a calibration question: the lever and the class partition are the same
   axis.

## Conclusions

**On the frozen ADI classes the two-stage model is already conditionally unbiased,
so the reallocation rule cannot be justified as a bias correction. Under a
calibration constraint it is worth $t^\*=0.10$ — a real but modest −0.46 pp at the
row grain, validated out-of-sample, with every grain improving and every class
bias left inside noise. The $t=0.60$ operating point selected by row WAPE in the
previous revision is rejected: it manufactures a −14% bias on the sparsest class.**

1. **The premise failed the test.** Aggregate class biases are A1 −1.20%,
   A2 +1.29%, A3 +3.84% against standard errors of 2.71 / 3.89 / 3.67 — |t| ≤ 1.04,
   none distinguishable from zero. The per-origin values swing −22% to +42% and
   move *together* across classes, which is the event/post-event weekly-total
   error `09_01` Addendum 3 already attributed to the calendar programme, not a
   class-specific tilt.
2. **Why WAPE could not choose $t$.** Under the global scope the renormalisation
   pins pooled bias exactly, so the sweep's bias column is constant by
   construction and the only remaining signal is row WAPE — which falls
   monotonically to $t≈0.60$. A criterion whose constraint cannot bind is not a
   criterion. The previous revision's $t^\*=0.60$ is what that produces.
3. **Calibration-constrained selection.** Taking the largest $t$ whose *induced*
   change in class bias stays within one tune-half standard error (A1 4.73%,
   A2 7.09%, A3 5.97%) gives $t^\*=0.10$; $t=0.15$ already moves A3 by −5.98%.
   The induced change is a paired per-origin difference, so it is estimated far
   more tightly than the levels — this constraint bites.
4. **Held-out result at $t^\*=0.10$.** Origins 11–20, frozen: row
   **56.19% vs 56.65%** (−0.46 pp), article-store-week 35.17% vs 35.53% (−0.37),
   article-day 22.76% vs 22.85% (−0.09), article-week 18.52% vs 18.60% (−0.08).
   Every grain improves. Class biases move A1 −2.03% → −1.47%, A2 −0.16% → −0.89%,
   A3 +1.66% → −1.54% — all still inside one standard error. All 20 origins
   improve individually (−0.60 to −0.25 pp). For scale: the weather programme
   bought −0.77 pp and the `08_04` campaign batch −0.53 pp, both by refitting.
5. **What $t=0.60$ actually costs.** Row 54.52% (−2.13 pp held out) but
   article-day +2.50 pp, article-week +2.72 pp, and class bias A2 −8.77%,
   A3 −16.15%. In-sample that is −2.32 and −3.88 standard errors. The hard
   threshold at $t=0.40$ is marginally better on row (54.38%) and marginally
   worse on calibration (A3 −17.30%) — the two rules agree about the trade, they
   differ only in where along it they sit.
6. **Mechanism at $t^\*$.** 30.6% of rows are down-weighted but only **0.61% of
   forecast mass** moves (carrying 0.55% of realized volume), renormalising by
   ×1.0062. The gain is bought by removing mass from rows that were almost surely
   zero, not by re-shaping the forecast.
7. **The gain lives across series, not within weeks.** The article-store-week
   scope is nearly inert: 0.27% of mass is displaceable there versus 0.61%
   globally, and the weekday gradient (Sat ≈ 2× Mon) is already priced in.
   Consistent with the `09_01` allocation ladder. This is also why $t$ is a
   calibration parameter: the only axis the rule can move along is the same axis
   the classes are defined on.
8. **Relation to the floors.** 56.2% remains far above the 44.9%
   perfect-occurrence bound (`09_01`). The rule harvests a sliver of the
   *decision-theoretic* gap — the metric's median-seeking geometry — not new
   information. It is a decision layer for the row-grain KPI, not a better
   forecast of demand, and at $t^\*$ it is a deliberately small one.
9. **Caveats.**
   - The constraint is only as good as the partition. ADI terciles are frozen
     before the first origin and are the right kind of variable, but they are one
     partition among many; a rule that is calibration-neutral on ADI classes is
     not thereby neutral on promotion state, weekday, or store. Those were not
     tested here.
   - Class membership is frozen for all 20 origins. A production layer would
     recompute it, at the cost of series changing class mid-evaluation.
   - The exponent is fixed at 2 throughout; tuning it jointly with $t$ was not
     explored and would need its own held-out protocol.
   - The `unclassified` band (0.4% of volume) carries a +14.15% baseline bias that
     none of this addresses.